# Import packages

In [ ]:
from json.decoder import NaN

import pandas as pd
from seaborn import lineplot, barplot
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
from visualisierungen import *

# Read csv

In [ ]:
df = pd.read_csv("../data/processed/df_with_positions.csv")
print(df.head())

## Datatype Adaption for this particular analysis

In [ ]:
partei_cols = ['zustimmung_p-fdp', 'zustimmung_p-sps', 'zustimmung_p-svp', 'zustimmung_p-mitte', 'zustimmung_p-glp', 'zustimmung_p-gps']

institutionen = {
    'Bundesrat': ['br-pos_label'],
    'Bundesversammlung': ['bv-pos_label'],
    'Parteien': ['p-fdp_label', 'p-sps_label', 'p-svp_label', 'p-mitte_label', 'p-glp_label', 'p-gps_label']
}

# Allgemeine zeitliche Analysen

### Balkendiagramm: Anzahl Abstimmungen über Zeit

In [ ]:
anzahl_vote = df.groupby(['jahrzehnt', 'rechtsform_name']).size().reset_index(name='anzahl')
print(anzahl_vote)

pivot = anzahl_vote.pivot(index='jahrzehnt', columns='rechtsform_name', values='anzahl').fillna(0)
fig = gestapeltes_balkendiagramm(
    pivot,
    xlabel="", ylabel="Anzahl", figsize=(12, 6))

plt.savefig("../Blog/blog_plots/d4_abstimmungen_zeit.png", dpi=300, bbox_inches='tight')
fig


### Balkendiagramm (Anteil): Wer gab Abstimmungsempfehlungen wann?

In [ ]:
# Zähler: Klare Empfehlung abgegeben
gezaehlt = ["Befürwortend", "Ablehnend", "Leer einlegen", "Vorzug Gegenentwurf", "Vorzug Volksinitiative"]

# Nenner: Alles wo das Organ existierte (inkl. keine Empfehlung)
existiert_labels = [
    "Befürwortend",          # 1  → Empfehlung ✓
    "Ablehnend",             # 2  → Empfehlung ✓
    "Keine Empfehlung",      # 3  → existiert, aber keine konkrete Empfehlung ✗
    "Leer einlegen",         # 4  → Empfehlung ✓
    "Stimmfreigabe",         # 5  → existiert, aber keine konkrete Empfehlung ✗
    "Vorzug Gegenentwurf",   # 8  → Empfehlung ✓
    "Vorzug Volksinitiative",# 9  → Empfehlung ✓
    "Neutral",               # 66 → existiert, aber keine konkrete Empfehlung ✗
]
# "Existiert nicht" (9999) und NaN → aus beiden ausgeschlossen

frames = []
for gruppe, cols in institutionen.items():
    if cols in ("br-pos_label", "bv-pos_label"):
        # Nur hier neue Logik: Anteil non-NaN --> BR und BV hat es schon immer gegeben
        anteil = df.groupby('jahrzehnt')[cols].apply(lambda x: x.notna().mean())
        temp = anteil.reset_index(name='anteil')
    else:
        # Hier Non-Nan ausschliessen, da es teilweise Parteien nicht gab
        empfehlungen = df.groupby('jahrzehnt')[cols].apply(
            lambda x: x.isin(gezaehlt).sum().sum()
        )
        existiert = df.groupby('jahrzehnt')[cols].apply(
            lambda x: x.isin(existiert_labels).sum().sum()
        )
        temp = (empfehlungen / existiert).reset_index(name='anteil')
    temp['gruppe'] = gruppe
    frames.append(temp)

numbers_vote = pd.concat(frames, ignore_index=True)
print(numbers_vote[(numbers_vote['gruppe'] == "Parteien") & (numbers_vote['jahrzehnt'].astype(int) > 1890)]['anteil'].mean())

In [ ]:
balkendiagramm(numbers_vote, x='jahrzehnt', y='anteil', hue='gruppe', palette=PALETTE_KATEGORIAL,
               titel="Anteil abgegebener Empfehlungen pro Jahrzehnt",
               xlabel="Jahrzehnt", ylabel="Anteil",
               figsize=(12, 6), rotation=45)
plt.savefig("../Blog/blog_plots/d1_empfehlungen_zeit.png", dpi=300, bbox_inches='tight')
fig

In [ ]:
df

# Individuelle Entwicklung

## Postion Insitution vs. Position Bevölkerung

### BR

#### Liniendiagramm: BR vs. Bevölkerung

In [ ]:
liniendiagramm(df, x="legisjahr", y="zustimmung_br-pos", xlabel="Jahr", ylabel="Übereinstimmung", rotation=90, errorbar=True, hline=0)

#### Boxplots+Lineplot: BR vs. Bevölkerung

In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))

# Kurzzeit: Boxplot pro Jahr
sns.boxplot(data=df, x="jahr", y="zustimmung_br-pos", ax=ax, color=HAUPTFARBE)

# Langzeit: Mittelwert pro Legislaturperiode als Linie
legis_mean = df.groupby("legisjahr")["zustimmung_br-pos"].mean().reset_index()

# Mapping: jedem Legislaturjahr den mittleren x-Position zuordnen
jahr_labels = [t.get_text() for t in ax.get_xticklabels()]
legis_x = []
legis_y = []
for _, row in legis_mean.iterrows():
    matching = df[df["legisjahr"] == row["legisjahr"]]["jahr"]
    mid_jahr = str(int(matching.mean()))
    if mid_jahr in jahr_labels:
        legis_x.append(jahr_labels.index(mid_jahr))
        legis_y.append(row["zustimmung_br-pos"])

ax.plot(legis_x, legis_y, color=AKZENTFARBE, linewidth=2, marker="o", label="Durchschnittliche Zustimmung mit Bundesrat pro Legislatur")

ax.axhline(0, color="grey", linestyle="--")
ax.set_xlabel("Jahr")
ax.set_ylabel("Übereinstimmung")
ax.tick_params(axis="x", rotation=90)

# x-Achse ab 1848 begrenzen (Index in den kategorialen Labels finden)
ax.set_xlim(left="1848", right="2025")
ax.set_ylim(bottom=-0.5, top=0.5)


ax.legend()
plt.tight_layout()
plt.show()

### BV

#### Liniendiagramm: BV vs. Bevölkerung

In [ ]:
liniendiagramm(df, x="5_jahre", y="zustimmung_bv-pos", xlabel="Jahr", ylabel="Übereinstimmung", rotation=90, errorbar=True,  hline=0)

#### Boxplots+Lineplot: BV vs. Bevölkerung

In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))

# Kurzzeit: Boxplot pro Jahr
sns.boxplot(data=df, x="jahr", y="zustimmung_bv-pos", ax=ax, color=HAUPTFARBE)

# Langzeit: Mittelwert pro Legislaturperiode als Linie
legis_mean = df.groupby("5_jahre")["zustimmung_bv-pos"].mean().reset_index()

# Mapping: jedem Legislaturjahr  x-Position zuordnen
jahr_labels = [t.get_text() for t in ax.get_xticklabels()]
legis_x = []
legis_y = []
for _, row in legis_mean.iterrows():
    matching = df[df["5_jahre"] == row["5_jahre"]]["jahr"]
    mid_jahr = str(int(matching.mean()))
    if mid_jahr in jahr_labels:
        legis_x.append(jahr_labels.index(mid_jahr))
        legis_y.append(row["zustimmung_bv-pos"])

ax.plot(legis_x, legis_y, color=AKZENTFARBE, linewidth=2, marker="o", label="Durchschnittliche Zustimmung mit Bundesversammlung pro Legislatur")

ax.axhline(0, color=AKZENTFARBE, linestyle="--")
ax.set_xlabel("Jahr")
ax.set_ylabel("Übereinstimmung")
ax.tick_params(axis="x", rotation=90)

# x-Achse ab 1848 begrenzen (Index in den kategorialen Labels finden)
ax.set_xlim(left="1848", right="2025")
ax.set_ylim(bottom=-0.5, top=0.5)


ax.legend()
plt.tight_layout()
plt.show()

### Parteien

In [ ]:
partei_cols = ["zustimmung_p-svp", "zustimmung_p-fdp", "zustimmung_p-mitte", "zustimmung_p-glp", "zustimmung_p-sps", "zustimmung_p-gps"]

df_long = df[["jahrzehnt"] + partei_cols].melt(
    id_vars="jahrzehnt", var_name="partei", value_name="zustimmung")
df_long['partei'] = df_long['partei'].str.replace('zustimmung_p-', '').str.upper()

liniendiagramm(df_long, x="jahrzehnt", y="zustimmung", hue="partei",
               xlabel="Jahrzehnt", ylabel="Übereinstimmung", rotation=90, hline=0)


#### Boxplot+Lineplot: Individuelle Parteien

In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))

# Kurzzeit: Boxplot pro Jahr
sns.boxplot(data=df, x="jahr", y="zustimmung_p-sps", ax=ax, color=HAUPTFARBE)

# Langzeit: Mittelwert pro Legislaturperiode als Linie
legis_mean = df.groupby("5_jahre")["zustimmung_p-sps"].mean().reset_index()

# Mapping: jedem Legislaturjahr  x-Position zuordnen
jahr_labels = [t.get_text() for t in ax.get_xticklabels()]
legis_x = []
legis_y = []
for _, row in legis_mean.iterrows():
    matching = df[df["5_jahre"] == row["5_jahre"]]["jahr"]
    mid_jahr = str(int(matching.mean()))
    if mid_jahr in jahr_labels:
        legis_x.append(jahr_labels.index(mid_jahr))
        legis_y.append(row["zustimmung_p-sps"])

ax.plot(legis_x, legis_y, color=AKZENTFARBE, linewidth=2, marker="o", label="Durchschnittliche Zustimmung mit SP pro Legislatur")

ax.axhline(0, color=AKZENTFARBE, linestyle="--")
ax.set_xlabel("Jahr")
ax.set_ylabel("Übereinstimmung")
ax.tick_params(axis="x", rotation=90)

# x-Achse ab 1848 begrenzen (Index in den kategorialen Labels finden)
ax.set_xlim(left="1848", right="2025")
ax.set_ylim(bottom=-0.5, top=0.5)


ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(df['5_jahre'].value_counts().sort_index())

# Vergleich zwischen BR, BV, Parteien

In [ ]:
bund = df[['legisjahr', 'zustimmung_br-pos', 'zustimmung_bv-pos'] + partei_cols].copy()
bund['zustimmung_parteien'] = bund[partei_cols].mean(axis=1)

bund = bund[['legisjahr', 'zustimmung_br-pos', 'zustimmung_bv-pos', 'zustimmung_parteien']].melt(
    id_vars='legisjahr', var_name='institution', value_name='zustimmung')

liniendiagramm(bund, x="legisjahr", y="zustimmung", hue="institution",
               xlabel="Legislatur", ylabel="Übereinstimmung", rotation=45, hline=0)

Dieser Plot ist spannend. Ab den 1970er Jahren ist die Zustimmung der Bevölkerung mit Bund/Parlament sehr viel höher als mit den Parteien. Ausserdem BV und BR sehr deckungsgleich ab 1980?
Gab es einen instiutionellen Wandel der diese Entwicklung erklärt? Einfluss Frauenstimmrecht? Später Aufschwung SVP? Uniforme Position
Einführung Zauberformel 1959 --> Erhöhte Einigkeit zwischen BV und BR

# Rechtsform

## BR

In [ ]:
scatterplot(df, x="jahrzehnt", y="zustimmung_br-pos", size=None, titel="Zustimmung mit Bundesrat nach Reechtsform", xlabel="", ylabel="", legendentitel='Rechtsform',
               sizes=(20,800), alpha=0.8, hue='rechtsform_name', figsize=(10,5), rotation=90)

In [ ]:
liniendiagramm(df, x="jahrzehnt", y="zustimmung_br-pos",
               xlabel="Jahrzehnt", ylabel="Übereinstimmung mit Bundesversammlung", rotation=90, hue="rechtsform_name", errorbar=False, hline=0)

## BV

In [ ]:
liniendiagramm(df, x="jahrzehnt", y="zustimmung_bv-pos",
               xlabel="Jahr", ylabel="Übereinstimmung", rotation=90, hue="rechtsform_name", errorbar=None,  hline=0)

#### Legislaturjahre

In [ ]:
rechtsform = df.groupby('jahrzehnt')['rechtsform_name'].value_counts(normalize=True).reset_index(name='anzahl')

neue_zeilen = pd.DataFrame({
    'jahrzehnt': [1850],
    'rechtsform_name': ['Keine Abstimmungen'],
    'anzahl': [0]
})

rechtsform = pd.concat([rechtsform, neue_zeilen], ignore_index=True).sort_values('jahrzehnt').reset_index(drop=True)

In [ ]:
balkendiagramm(data=rechtsform, x='jahrzehnt', y='anzahl', hue='rechtsform_name', palette=PALETTE_KATEGORIAL)

In [ ]:
df

# Erste Schritte Interaktive Grafik

In [ ]:
import plotly.io as pio
pio.renderers.default = "browser"
print(pio.renderers.default)

In [ ]:
akteur_cols = [
    'zustimmung_br-pos', 'zustimmung_bv-pos',
    'zustimmung_p-gps',    # Grüne
    'zustimmung_p-sps',    # SP
    'zustimmung_p-glp',    # GLP
    'zustimmung_p-mitte',  # Mitte
    'zustimmung_p-fdp',    # FDP
    'zustimmung_p-svp',    # SVP
]

label_map = {
    'zustimmung_br-pos':  'Bundesrat',
    'zustimmung_bv-pos':  'Bundesversammlung',
    'zustimmung_p-gps':   'Grüne',
    'zustimmung_p-sps':   'SP',
    'zustimmung_p-glp':   'GLP',
    'zustimmung_p-mitte': 'Mitte',
    'zustimmung_p-fdp':   'FDP',
    'zustimmung_p-svp':   'SVP',
}

farben_map = {
    'Bundesrat':         '#4477AA',
    'Bundesversammlung': '#CC6677',
    'Grüne':  '#999933',
    'SP':     '#882255',
    'GLP':    '#44AA99',
    'Mitte':  '#DDCC77',
    'FDP':    '#332288',
    'SVP':    '#117733',
}

fig = liniendiagramm_interaktiv_zeitwahl(
    df,
    wert_cols=akteur_cols,
    label_map=label_map,
    farben_map=farben_map,
    titel='Übereinstimmung pro Akteur',
    xlabel='Jahr',
    ylabel='Übereinstimmung',
    yrange=(-0.5, 0.5))

fig.show()
fig.write_html(
    "../Blog/blog_plots/d3_kongruenz_zeit.html",
    include_plotlyjs='inline',
    full_html=True,
)